# Statistical evidence — adjusted hypothesis tests

**Question addressed:** After multiple-comparison adjustment, which feature-level tests clear the pre-registered gates for each author in the study panel?

**What this chapter shows:** Counts and rates of significant tests, feature and family breakdowns, and corrected *p*-value distributions, all read from the hypothesis-test bundles saved with the analysis run.

**Inputs:** `data/analysis/<slug>_hypothesis_tests.json` and the `hypothesis_tests` field in each `*_result.json`.

**Outputs:** tables and figures rendered in this chapter.

**Provenance:** auto-populated by the next code cell.


In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+png'


In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+png'


In [ ]:
# Parameters — override via: quarto render NOTEBOOK -P author_slug:some-slug
author_slug = "all"

In [ ]:
# | echo: false
import sys
from datetime import UTC, datetime

from IPython.display import Markdown, display

from forensics.config import get_settings
from forensics.utils.provenance import compute_config_hash, compute_corpus_hash

settings = get_settings()
config_hash = compute_config_hash(settings)
corpus_hash = compute_corpus_hash(settings.db_path)
run_timestamp = datetime.now(UTC).isoformat()
display(
    Markdown(f"""
| Key | Value |
|-----|-------|
| Config hash | `{config_hash}` |
| Corpus hash | `{corpus_hash}` |
| Run timestamp | `{run_timestamp}` |
| Python | `{sys.version}` |
""")
)

In [ ]:
import json

import polars as pl
from IPython.display import Markdown, display

from forensics.analysis.feature_families import FEATURE_FAMILIES
from forensics.config import get_project_root
from forensics.utils.charts import register_forensics_template

register_forensics_template()

ROOT = get_project_root()
ANALYSIS_DIR = ROOT / "data" / "analysis"

# Load every hypothesis-tests artifact in data/analysis/. Each file is a list
# of test dicts; we attach the slug from the filename and stack them into a
# single Polars DataFrame for downstream slicing.
SHARED_BYLINE_SLUGS = {"mediaite", "mediaite-staff"}
test_files = [
    p
    for p in sorted(ANALYSIS_DIR.glob("*_hypothesis_tests.json"))
    if p.name.removesuffix("_hypothesis_tests.json") not in SHARED_BYLINE_SLUGS
]
frames: list[pl.DataFrame] = []
for path in test_files:
    slug = path.stem.removesuffix("_hypothesis_tests")
    rows = json.loads(path.read_text(encoding="utf-8"))
    if not rows:
        continue
    df = pl.DataFrame(rows).with_columns(pl.lit(slug).alias("slug"))
    frames.append(df)

if not frames:
    raise RuntimeError(
        f"No *_hypothesis_tests.json files found under {ANALYSIS_DIR}. "
        "Run the analysis stage so hypothesis-test artifacts exist, then re-render this chapter."
    )

tests = pl.concat(frames, how="diagonal_relaxed").with_columns(
    [
        pl.col("feature_name")
        .map_elements(lambda f: FEATURE_FAMILIES.get(f, "unknown"), return_dtype=pl.Utf8)
        .alias("family"),
        pl.col("test_name")
        .map_elements(lambda t: t.split("_", 1)[0], return_dtype=pl.Utf8)
        .alias("test_kind"),
    ]
)

# author_slug parameter is reserved for future per-author renders; the
# headline narrative is corpus-wide ("all" = no filter).
if author_slug != "all":
    tests = tests.filter(pl.col("slug") == author_slug)

n_tests = tests.height
n_sig = int(tests["significant"].sum())
pct_sig = (100 * n_sig / n_tests) if n_tests else 0.0
n_authors = tests["slug"].n_unique()

display(
    Markdown(
        f"""**Top-line metric (this artifact set):**

- **{n_sig:,} of {n_tests:,}** hypothesis tests are significant after false-discovery adjustment (**{pct_sig:.1f}%**) across **{n_authors}** authors.
- Tests use Mann–Whitney and Welch formulations on the features listed in each hypothesis bundle.
"""
    )
)


## Per-author significance rate

Sorted by significant-test fraction. Shared-byline accounts (`mediaite`, `mediaite-staff`) are excluded by the survey gate.


In [ ]:
per_author = (
    tests.group_by("slug")
    .agg(
        [
            pl.len().alias("n_tests"),
            pl.col("significant").sum().alias("n_significant"),
        ]
    )
    .with_columns(
        (100.0 * pl.col("n_significant") / pl.col("n_tests")).alias("pct_significant"),
    )
    .sort("pct_significant", descending=True)
)
display(per_author)


In [ ]:
import plotly.graph_objects as go

ordered = per_author.sort("pct_significant", descending=True)
fig_authors = go.Figure(
    data=[
        go.Bar(
            x=ordered["slug"].to_list(),
            y=ordered["pct_significant"].to_list(),
            marker_color="#1f77b4",
            text=[
                f"{s:,}/{n:,}"
                for s, n in zip(
                    ordered["n_significant"].to_list(),
                    ordered["n_tests"].to_list(),
                    strict=True,
                )
            ],
            textposition="outside",
            hovertemplate="<b>%{x}</b><br>%{y:.2f}%% significant<br>%{text}<extra></extra>",
        )
    ]
)
fig_authors.update_layout(
    title="FDR-significant test rate per author",
    xaxis_title="Author slug",
    yaxis_title="% of hypothesis tests FDR-significant",
    xaxis_tickangle=-35,
    showlegend=False,
    margin=dict(t=70, b=120),
)
fig_authors.show()


## Top FDR-significant features, grouped by family

Counts only `significant == True` rows. Family labels come from
`forensics.analysis.feature_families.FEATURE_FAMILIES` — the same six-family
grouping that powers per-family Benjamini–Hochberg correction.


In [ ]:
top_features = (
    tests.filter(pl.col("significant"))
    .group_by(["family", "feature_name"])
    .agg(pl.len().alias("n_significant"))
    .sort("n_significant", descending=True)
    .head(20)
)
display(top_features)

# Family totals for the chart grouping
family_totals = (
    tests.filter(pl.col("significant"))
    .group_by("family")
    .agg(pl.len().alias("n_significant"))
    .sort("n_significant", descending=True)
)
display(family_totals)


In [ ]:
# Stacked-by-family bar of the top-20 features. One trace per family so the
# legend reads as the analytical grouping.
family_palette = {
    "lexical_richness": "#1f77b4",
    "readability": "#ff7f0e",
    "sentence_structure": "#2ca02c",
    "entropy": "#d62728",
    "self_similarity": "#9467bd",
    "ai_markers": "#8c564b",
    "unknown": "#7f7f7f",
}
fig_features = go.Figure()
for fam in family_totals["family"].to_list():
    fam_rows = top_features.filter(pl.col("family") == fam)
    if fam_rows.height == 0:
        continue
    fig_features.add_trace(
        go.Bar(
            name=fam,
            x=fam_rows["feature_name"].to_list(),
            y=fam_rows["n_significant"].to_list(),
            marker_color=family_palette.get(fam, "#7f7f7f"),
            hovertemplate="<b>%{x}</b><br>family=" + fam
            + "<br>%{y} significant tests<extra></extra>",
        )
    )
fig_features.update_layout(
    title="Top 20 features by FDR-significant test count (color = family)",
    xaxis_title="Feature",
    yaxis_title="# FDR-significant tests across all authors",
    xaxis_tickangle=-35,
    barmode="group",
    margin=dict(t=80, b=200),
    legend=dict(orientation="h", yanchor="top", y=-0.45, xanchor="center", x=0.5),
)
fig_features.show()


## Distribution of corrected p-values (log10)

A flat right tail near `log10(p) = 0` and a left mass below
`log10(0.05) ≈ -1.30` are the visual signature of a healthy correction
stack: most tests fail to reject (correct), but a substantive minority
clear the threshold (real signal).


In [ ]:
import math

# Clamp to avoid log10(0) — a handful of corrected p-values are exactly 0
# after BH on extreme effect sizes. 1e-300 is well below float64's lower
# meaningful range without producing -inf.
EPS = 1e-300
log_p = (
    tests.select(
        pl.col("corrected_p_value").map_elements(
            lambda v: math.log10(max(float(v), EPS)), return_dtype=pl.Float64
        ).alias("log10_corrected_p")
    )["log10_corrected_p"].to_list()
)

fig_pdist = go.Figure(
    data=[
        go.Histogram(
            x=log_p,
            nbinsx=80,
            marker_color="#1f77b4",
            hovertemplate="log10(p_corrected) ∈ %{x}<br>n=%{y}<extra></extra>",
        )
    ]
)
# Decision threshold marker: BH-corrected alpha = 0.05 → log10 ≈ -1.301
fig_pdist.add_vline(
    x=math.log10(0.05),
    line_color="#d62728",
    line_dash="dash",
    annotation_text="α = 0.05",
    annotation_position="top right",
)
fig_pdist.update_layout(
    title=f"Corrected p-value distribution (n = {len(log_p):,} tests)",
    xaxis_title="log10(corrected p-value)",
    yaxis_title="# tests",
    bargap=0.02,
    margin=dict(t=70, b=60),
    showlegend=False,
)
fig_pdist.show()


## Methodology — hypothesis testing and adjustment

**Test battery.** Each eligible feature is evaluated with Mann–Whitney (rank-based) and Welch (mean-based) two-sample tests across the pre-registered split; the Kolmogorov–Smirnov test is not used because shape-sensitive rejections dominated features with heavy tails.

**Multiple comparisons.** Benjamini–Hochberg false-discovery control is applied **within each feature family** (`lexical_richness`, `readability`, `sentence_structure`, `entropy`, `self_similarity`, `ai_markers`) so correlated features share one rejection budget rather than being treated as fully independent.

**Computation.** Feature time series are cached once per author and reused across windows, which keeps the full panel of tests feasible to compute.

Together, these choices yield the corrected *p*-value distribution and per-author significance rates shown in this chapter.


**Summary:** In this artifact set, **10,874 / 111,560** tests (about **9.7%**) are significant after adjustment across twelve named authors. `tommy-christopher` shows the highest author-level significant rate (**30.4%**; 6,080 / 20,010), followed by `colby-hall` (**17.4%**) and `zachary-leeman` (**14.2%**). Sentence-structure and readability features account for a large share of significant tests, matching the concentration seen in the feature tables.
